# Notebook 07: Reranking & RAG Evaluation

**Time:** ~35 minutes  
**Goal:** Add a reranker (the highest-leverage 100ms in RAG), then evaluate end-to-end with RAGAS-style metrics.

This notebook will:
1. Add a **cross-encoder reranker** (BGE / MS-MARCO MiniLM) on top of hybrid retrieval
2. Try **FlashRank** (lightweight, no torch) for a CPU-friendly alternative
3. Implement four RAGAS-style **reference-free** metrics:
   - faithfulness (are answer claims grounded?)
   - answer_relevancy (does the answer address the question?)
   - context_precision (are retrieved chunks actually relevant?)
   - context_recall (does retrieval cover a reference answer?)
4. Run the suite on YOUR pipeline and identify failure modes
5. Compare two configurations and report the delta

> **Production point:** rerankers cost ~100ms but typically deliver 30-40% recall improvements. Cheap quality.


## Setup + build pipeline


In [1]:
import os, sys, time, importlib, json
from pathlib import Path

notebook_dir = os.getcwd()
parent_dir   = str(Path(notebook_dir).parent)
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)

from dotenv import load_dotenv
load_dotenv(os.path.join(parent_dir, '.env'), override=True)

import src.llm_client, src.cost_tracker, src.utils, src.config
import src.document_loader, src.chunking, src.embeddings, src.vector_store
import src.retrieval, src.reranker, src.rag_evaluation, src.rag_pipeline
for mod in [src.llm_client, src.cost_tracker, src.utils, src.config,
            src.document_loader, src.chunking, src.embeddings, src.vector_store,
            src.retrieval, src.reranker, src.rag_evaluation, src.rag_pipeline]:
    importlib.reload(mod)

from src.llm_client import LLMClient
from src.cost_tracker import CostTracker
from src.utils import format_response, append_to_reflection
from src.document_loader import load_directory
from src.chunking import recursive_chunk
from src.embeddings import EmbeddingModel
from src.vector_store import FAISSStore
from src.retrieval import HybridRetriever, BM25Retriever
from src.reranker import CrossEncoderReranker, FlashRankReranker
from src.rag_evaluation import (
    faithfulness, answer_relevancy, context_precision, context_recall,
    evaluate_rag,
)
from src.rag_pipeline import RAGPipeline
import src.config as config

import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

client  = LLMClient(path=config.PATH)
tracker = CostTracker()

outputs_dir = os.path.join('..', 'outputs')
test_data   = os.path.join('..', 'test_data')

print('Setup complete -- ready for Notebook 07')


✓ Claude API client initialized
  Default model: claude-sonnet-4-6
  Available: claude-sonnet-4-6, claude-opus-4-6, claude-haiku-4-5-20251001
Setup complete -- ready for Notebook 07


In [2]:
# Build hybrid pipeline (no reranker yet)
my_docs = load_directory(os.path.join(test_data, 'my_corpus'))
chunks = []
for d in my_docs:
    for i, c in enumerate(recursive_chunk(d['text'], chunk_size=500, overlap=50)):
        chunks.append({'text': c, 'metadata': {'source': os.path.basename(d['source']), 'chunk_id': i}})
print(f'{len(chunks)} chunks')

em = EmbeddingModel('all-MiniLM-L6-v2')
vecs = em.encode([c['text'] for c in chunks])
fs = FAISSStore(dim=em.dim); fs.add(chunks, vecs)
bm25 = BM25Retriever(chunks)
hybrid = HybridRetriever(fs, em, bm25)


  ✓ Loaded portfolio_notes.txt: 2,669 chars
  ✓ Loaded sample_resume.pdf: 1 pages, 2,082 chars (via pymupdf)

✓ Loaded 2 documents from ..\test_data\my_corpus
  ✓ recursive_chunk: 8 chunks (size~500, overlap=50)
  ✓ recursive_chunk: 5 chunks (size~500, overlap=50)
  ✓ recursive_chunk: 5 chunks (size~500, overlap=50)
13 chunks
  Loading all-MiniLM-L6-v2 (sentence-transformers, cpu)...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
c:\Users\lflyl\OneDrive\文档\inferenceai\week4\Homework4-Submission\src\embeddings.py:83: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  self._dim = self._client.get_sentence_embedding_dimension()


  ✓ Loaded in 1.2s, dim=384
  ✓ FAISS: 13 vectors added (total: 13)
  ✓ BM25 indexed 13 chunks


## Part 1 — Cross-encoder reranker (sentence-transformers)

Cross-encoders are slow but accurate: they jointly score `(query, doc)` instead of computing dot products in shared space. A 6-layer MiniLM reranker scores ~100 docs/sec on CPU.


In [3]:
print('=' * 65)
print('Experiment 1: Hybrid -> Cross-encoder reranker')
print('=' * 65)

ce_reranker = CrossEncoderReranker('cross-encoder/ms-marco-MiniLM-L-6-v2')

QUERY = 'What machine learning frameworks does this candidate know?'
candidates = hybrid.search(QUERY, k=20)  # over-fetch
reranked   = ce_reranker.rerank(QUERY, candidates, top_k=5)

print('\nBefore reranker (top 5 of 20):')
for r in candidates[:5]:
    print(f'  [{r["score"]:.3f}] {r["metadata"]["source"]}::{r["metadata"]["chunk_id"]}')
print('\nAfter reranker (top 5):')
for r in reranked:
    print(f'  [rerank={r["rerank_score"]:+.3f}] {r["metadata"]["source"]}::{r["metadata"]["chunk_id"]}')


Experiment 1: Hybrid -> Cross-encoder reranker
  Loading reranker cross-encoder/ms-marco-MiniLM-L-6-v2...


config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

c:\Users\lflyl\OneDrive\文档\inferenceai\week4\Homework4-Submission\.venv\lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\lflyl\.cache\huggingface\hub\models--cross-encoder--ms-marco-MiniLM-L-6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

  ✓ Reranker ready
  Hybrid: dense=13 sparse=0 -> fused=13

Before reranker (top 5 of 20):
  [0.016] sample_resume.pdf::0
  [0.016] portfolio_notes.txt::0
  [0.016] portfolio_notes.txt::4
  [0.016] sample_resume.pdf::1
  [0.015] portfolio_notes.txt::7

After reranker (top 5):
  [rerank=-10.912] sample_resume.pdf::0
  [rerank=-11.027] portfolio_notes.txt::7
  [rerank=-11.077] portfolio_notes.txt::0
  [rerank=-11.258] portfolio_notes.txt::5
  [rerank=-11.263] sample_resume.pdf::2


## Part 2 — FlashRank (CPU-friendly, no torch)


In [4]:
print('=' * 65)
print('Experiment 2: FlashRank (lightweight)')
print('=' * 65)

try:
    fr = FlashRankReranker('ms-marco-TinyBERT-L-2-v2')
    fr_reranked = fr.rerank(QUERY, candidates, top_k=5)
    print('\nFlashRank top-5:')
    for r in fr_reranked:
        print(f'  [{r["rerank_score"]:.3f}] {r["metadata"]["source"]}::{r["metadata"]["chunk_id"]}')
except Exception as e:
    print(f'FlashRank not available ({e}). pip install flashrank to enable.')


INFO:flashrank.Ranker:Downloading ms-marco-TinyBERT-L-2-v2...


Experiment 2: FlashRank (lightweight)


ms-marco-TinyBERT-L-2-v2.zip: 100%|██████████| 3.26M/3.26M [00:00<00:00, 107MiB/s]


  ✓ FlashRank ready (ms-marco-TinyBERT-L-2-v2)

FlashRank top-5:
  [0.000] portfolio_notes.txt::7
  [0.000] portfolio_notes.txt::6
  [0.000] portfolio_notes.txt::5
  [0.000] sample_resume.pdf::2
  [0.000] portfolio_notes.txt::0


## Part 3 — End-to-end RAG with reranker

Wire it all together via `RAGPipeline` and ask a real question.


In [5]:
rag = RAGPipeline(
    embedding_model=em,
    vector_store=fs,
    llm_client=client,
    chunker=lambda t: recursive_chunk(t, 500, 50),
    retriever=hybrid,
    reranker=ce_reranker,
    retrieve_k=20, rerank_k=4,
)

result = rag.answer(QUERY, max_tokens=400)
print('=' * 65); print(f'Q: {QUERY}'); print('=' * 65)
print('\nA:'); print(result['answer'])
print('\nSources:', result['sources'])
if 'error' not in result['raw_response']:
    tracker.add_call(result['raw_response'])


  Hybrid: dense=13 sparse=0 -> fused=13


INFO:httpx:HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"


Q: What machine learning frameworks does this candidate know?

A:
Based on the retrieved context, there is no mention of any machine learning frameworks in the candidate's skills, resume, or portfolio notes.

I don't know — that information isn't in my sources.

The candidate's listed skills include HTML/CSS, SQL (PostgreSQL, Oracle), JavaScript (Angular), Python (Django), REST APIs (GraphQL), and AWS (Redshift, S3), but no machine learning frameworks are referenced. [sample_resume.pdf, portfolio_notes.txt]

Sources: ['sample_resume.pdf', 'portfolio_notes.txt', 'portfolio_notes.txt', 'portfolio_notes.txt']


## Part 4 — Evaluate that answer with RAGAS-style metrics

All four metrics use Claude as the judge. Each is just an LLM call against a small prompt.


In [6]:
print('=' * 65)
print('Experiment 4: RAGAS-style evaluation')
print('=' * 65)

eval_result = evaluate_rag(
    llm_client=client,
    embedding_model=em,
    question=result['question'],
    answer=result['answer'],
    contexts=result['contexts'],
)

print(f'\n  Faithfulness:      {eval_result["faithfulness"]:.3f}')
print(f'  Answer relevancy:  {eval_result["answer_relevancy"]:.3f}')
print(f'  Context precision: {eval_result["context_precision"]:.3f}')

with open(os.path.join(outputs_dir, 'eval_result_07.json'), 'w') as f:
    # avoid raw_response (not JSON serializable)
    safe = {k: v for k, v in eval_result.items() if k != 'details'}
    safe['details'] = {
        m: {kk: vv for kk, vv in d.items() if kk != 'raw_response'}
        for m, d in eval_result['details'].items()
    }
    json.dump(safe, f, indent=2)


Experiment 4: RAGAS-style evaluation
  Evaluating: 'What machine learning frameworks does this candidate know?...'


INFO:httpx:HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.anthropic.co


  Faithfulness:      0.889
  Answer relevancy:  0.888
  Context precision: 0.000


## TODO 1 — A/B test: reranker on vs. off

Run the same 5 questions through two pipelines — one with no reranker, one with the cross-encoder reranker — and see whether the reranker actually helps.

### Cheat sheet: what the two metrics mean

| Metric | What it asks | Who the judge is | Range |
|---|---|---|---|
| **ContextPrecision** | *"Of the chunks we retrieved, how many are actually relevant to the question?"* | Claude reads each chunk and decides yes/no. | 0 = all noise, 1 = every chunk is on-topic |
| **Faithfulness** | *"Of the claims the model made in its answer, how many are actually supported by the retrieved chunks?"* | Claude extracts each factual claim and checks it against the chunks. | 0 = fully hallucinated, 1 = every claim is grounded |

### How to read the output

Per-question line looks like:
```
What programming languages ... : cp 0.64 → 1.00,  faith 0.88 → 1.00
                                  ↑ before   ↑ after    (higher = better)
```
- **`cp` = ContextPrecision**, **`faith` = Faithfulness**.
- `0.64 → 1.00` means: *no-rerank* got 0.64, *with-rerank* got 1.00 on that question. The reranker improved precision from "most chunks are relevant" to "all chunks are relevant."

### What a good reranker does

- **ContextPrecision should go up.** That's the reranker's whole job — promote relevant chunks, demote noise.
- **Faithfulness usually goes up or stays flat.** Better context → fewer hallucinations. But a reranker can *reduce* faithfulness if the new top chunks are tangential — the LLM then speculates to fill gaps. Watch for `faith ↓` + `cp ↑`: it means your reranker prefers topical-but-sparse chunks, and your LLM is making things up to compensate.
- **When a reranker is NOT worth it:** tiny corpora (<30 chunks — no headroom to rerank), budget/latency-critical paths (+100ms per query), or when the first-stage retriever is already near-perfect.


In [9]:
# TODO 1: A/B with vs. without reranker
import logging
logging.getLogger('httpx').setLevel(logging.WARNING)  # quiet the API-call spam

questions = [
    'What programming languages does this person know?',
    'What machine learning frameworks have they used?',
    'What relational databases do they know?',
    'What did they study?',
    'What was their role at their last job?',
]

rag_no_rerank = RAGPipeline(em, fs, client, chunker=lambda t: recursive_chunk(t,500,50),
                            retriever=hybrid, reranker=None, retrieve_k=4)
rag_with_rerank = rag

def arrow(before, after, tol=0.05):
    diff = after - before
    if abs(diff) < tol: return '='      # within noise band
    return '↑' if diff > 0 else '↓'

rows = []
print('Running 5 questions through both pipelines (each triggers ~8-12 judge calls)...\n')
for q in questions:
    a1 = rag_no_rerank.answer(q, max_tokens=350)
    a2 = rag_with_rerank.answer(q, max_tokens=350)
    if 'error' not in a1['raw_response']: tracker.add_call(a1['raw_response'])
    if 'error' not in a2['raw_response']: tracker.add_call(a2['raw_response'])

    # Score only context_precision + faithfulness to keep cost small
    cp_no  = context_precision(client, q, a1['contexts'])['score']
    cp_yes = context_precision(client, q, a2['contexts'])['score']
    f_no   = faithfulness(client, q, a1['answer'], a1['contexts'])['score']
    f_yes  = faithfulness(client, q, a2['answer'], a2['contexts'])['score']
    rows.append({'q': q[:50], 'cp_no': cp_no, 'cp_yes': cp_yes, 'f_no': f_no, 'f_yes': f_yes})

# --- Per-question readout ---------------------------------------------------
print('\n' + '=' * 82)
print('Per-question results  (before → after reranker; ↑ better, ↓ worse, = no change)')
print('=' * 82)
print(f'  {"Question":50s} | {"ContextPrec":^14s} | {"Faithfulness":^14s}')
print('  ' + '-' * 80)
for r in rows:
    cp_d = arrow(r['cp_no'], r['cp_yes'])
    f_d  = arrow(r['f_no'],  r['f_yes'])
    print(f'  {r["q"]:50s} | {r["cp_no"]:.2f} → {r["cp_yes"]:.2f} {cp_d} | {r["f_no"]:.2f} → {r["f_yes"]:.2f} {f_d}')

# --- Aggregate verdict ------------------------------------------------------
mean_cp_no  = sum(r['cp_no']  for r in rows) / len(rows)
mean_cp_yes = sum(r['cp_yes'] for r in rows) / len(rows)
mean_f_no   = sum(r['f_no']   for r in rows) / len(rows)
mean_f_yes  = sum(r['f_yes']  for r in rows) / len(rows)
cp_delta    = mean_cp_yes - mean_cp_no
f_delta     = mean_f_yes  - mean_f_no

print('\n' + '=' * 82)
print('Aggregate (mean across 5 questions)')
print('=' * 82)
print(f'  ContextPrecision (higher = fewer irrelevant chunks retrieved):')
print(f'    {mean_cp_no:.3f}  →  {mean_cp_yes:.3f}    Δ = {cp_delta:+.3f}')
print(f'  Faithfulness     (higher = fewer hallucinated claims in the answer):')
print(f'    {mean_f_no:.3f}  →  {mean_f_yes:.3f}    Δ = {f_delta:+.3f}')

def verdict(cp_d, f_d):
    if cp_d > 0.05 and f_d >= -0.02:
        return ('✓ Reranker HELPED. Precision went up and faithfulness held or improved — '
                'the reranker is promoting more relevant chunks and the LLM is grounding its '
                'answers better. Keep it in the pipeline.')
    if cp_d > 0.05 and f_d < -0.02:
        return ('⚠ MIXED RESULT. Precision up, but faithfulness went DOWN. The reranker is '
                'picking topical-but-thin chunks and your LLM is filling the gaps with '
                'speculation. Fix: raise retrieve_k so the reranker has more candidates, or '
                'add a faithfulness gate that refuses to answer when support is weak.')
    if abs(cp_d) <= 0.05 and abs(f_d) <= 0.05:
        return ('= Reranker had LITTLE EFFECT. Likely reason: your corpus is tiny (13 chunks), '
                'so top-k already covers most of the corpus and there is nothing meaningful to '
                'rerank. Try this again on the real project corpus in nb08.')
    if cp_d < -0.05:
        return ('✗ Reranker HURT precision. Likely cause: the cross-encoder was trained on '
                'a different domain (MS-MARCO web queries) than your corpus, or retrieve_k '
                'is so small the reranker has no good alternatives. Try a larger retrieve_k '
                'or a domain-matched reranker.')
    return '? Unusual pattern — inspect the per-question table above to see what moved.'

print('\nVerdict:')
print('  ' + verdict(cp_delta, f_delta))
print('=' * 82)

with open(os.path.join(outputs_dir, 'ab_reranker.json'), 'w') as f:
    json.dump({'rows': rows,
               'mean': {'cp_no': mean_cp_no, 'cp_yes': mean_cp_yes,
                        'f_no': mean_f_no, 'f_yes': mean_f_yes,
                        'cp_delta': cp_delta, 'f_delta': f_delta}},
              f, indent=2)

todo1_reflection = """
[YOUR REFLECTION]

- ContextPrecision delta: -0.017 (positive = reranker promoted more relevant chunks)
- Faithfulness delta: -0.117 (positive = answers got better grounded; negative = LLM is speculating more)
- Which question benefited most from the reranker? Why?
  "What programming languages does this person know?" showed the biggest improvement in context precision (0.64 → 1.00). The reranker successfully identified and promoted the resume skills section, giving the LLM a clear, focused source to ground its answer.

- Which question GOT WORSE? Inspect that question's contexts — did the reranker drop the
  chunk that actually contained the answer?
  "What did they study?" completely failed. Context precision dropped from 0.50 to 0.00, meaning the reranker excluded all relevant chunks. The cross-encoder, trained on web search queries, misunderstood the domain and promoted irrelevant sections, forcing the LLM to hallucinate an answer.

- Cost: the reranker adds ~100ms per query but no extra LLM calls. On a 13-chunk corpus it
  rarely earns its keep (nothing to rerank against). Would you ship it on your project
  corpus? Why or why not?
  On a 13-chunk corpus, I would NOT ship the reranker. The latency cost (+100ms) doesn't justify the quality loss (-0.117 faithfulness). The corpus is too small — there's nothing to meaningfully rerank. Rerankers shine with 1000+ documents; on 13 chunks, they often hurt.
  For the real project corpus in Notebook 08, I would absolutely use it. A larger corpus has enough irrelevant candidates for the reranker to filter, and the quality gains (30-40% recall improvement) outweigh the latency.
  
"""
print(todo1_reflection)


Running 5 questions through both pipelines (each triggers ~8-12 judge calls)...

  Hybrid: dense=13 sparse=1 -> fused=4
  Hybrid: dense=13 sparse=1 -> fused=13
  Hybrid: dense=13 sparse=1 -> fused=4
  Hybrid: dense=13 sparse=1 -> fused=13
  Hybrid: dense=13 sparse=1 -> fused=4
  Hybrid: dense=13 sparse=1 -> fused=13
  Hybrid: dense=13 sparse=0 -> fused=4
  Hybrid: dense=13 sparse=0 -> fused=13
  Hybrid: dense=13 sparse=7 -> fused=4
  Hybrid: dense=13 sparse=7 -> fused=13

Per-question results  (before → after reranker; ↑ better, ↓ worse, = no change)
  Question                                           |  ContextPrec   |  Faithfulness 
  --------------------------------------------------------------------------------
  What programming languages does this person know?  | 0.64 → 1.00 ↑ | 0.85 → 0.83 =
  What machine learning frameworks have they used?   | 0.00 → 0.00 = | 1.00 → 1.00 =
  What relational databases do they know?            | 1.00 → 0.83 ↓ | 0.88 → 0.67 ↓
  What did they st

## TODO 2 — Find a hallucination, design a guardrail


In [10]:
# TODO 2: Try to elicit a hallucination, then add a guardrail.

# Step 1: ask something whose answer is NOT in the corpus.
trick_q = 'What was this candidate\'s salary at their previous job?'
trick_a = rag_with_rerank.answer(trick_q, max_tokens=300)
if 'error' not in trick_a['raw_response']: tracker.add_call(trick_a['raw_response'])

print('Q:', trick_q); print('A:', trick_a['answer']); print('Sources:', trick_a['sources'])

score = faithfulness(client, trick_q, trick_a['answer'], trick_a['contexts'])
print(f'\nFaithfulness: {score["score"]:.2f} ({sum(score["supported"])}/{len(score["supported"])} claims supported)')

# Step 2: write a stricter system prompt that forces the model to refuse.
guarded = client.generate(
    prompt=f'Question: {trick_q}\n\nContext: {chr(10).join(trick_a["contexts"])}\n\nAnswer:',
    system='You answer ONLY using the provided context. If the answer is not present, reply EXACTLY: "That information is not in my sources." Cite [source] tags for every claim.',
    max_tokens=200, temperature=0.0,
)
if 'error' not in guarded: tracker.add_call(guarded)
print('\nGuarded answer:'); print(guarded.get('content', guarded.get('error')))

todo2_reflection = """
[YOUR REFLECTION]

- Did the unguarded model hallucinate? Cite the exact unsupported claim:
No hallucination detected. The unguarded model correctly refused: "I don't know — that information isn't in my sources. There is no mention of salary details anywhere in the retrieved context."
- Did the guarded prompt fix it? What edge cases break the guard?
The guarded prompt produced identical output: "That information is not in my sources." Since the unguarded model already refused correctly, the guard didn't need to intervene. Both responses are equally safe.
- A real production system would also add: refusal classifier (e.g. faithfulness gate, refusal classifier)
   - Require top-3 retrieved chunks to have min 0.8 relevance score
   - If below threshold, prepend: "Limited evidence for this answer"
"""
print(todo2_reflection)


  Hybrid: dense=13 sparse=8 -> fused=13
Q: What was this candidate's salary at their previous job?
A: I don't know — that information isn't in my sources. There is no mention of salary details anywhere in the retrieved context.
Sources: ['portfolio_notes.txt', 'portfolio_notes.txt', 'sample_resume.pdf', 'portfolio_notes.txt']

Faithfulness: 1.00 (2/2 claims supported)

Guarded answer:
That information is not in my sources.

[YOUR REFLECTION]

- Did the unguarded model hallucinate? Cite the exact unsupported claim:
No hallucination detected. The unguarded model correctly refused: "I don't know — that information isn't in my sources. There is no mention of salary details anywhere in the retrieved context."
- Did the guarded prompt fix it? What edge cases break the guard?
The guarded prompt produced identical output: "That information is not in my sources." Since the unguarded model already refused correctly, the guard didn't need to intervene. Both responses are equally safe.
- A real pr

## TODO 3 — Compare your favorite RAGAS metric to a manual judgment


In [12]:
# TODO 3: Score 3 (question, answer, contexts) tuples by HAND, then compare to the LLM judge.

manual_scores = []
for q in questions[:3]:
    res = rag_with_rerank.answer(q, max_tokens=300)
    if 'error' not in res['raw_response']: tracker.add_call(res['raw_response'])
    print('=' * 65); print(f'Q: {q}')
    print(f'A: {res["answer"][:400]}')
    print(f'Sources: {res["sources"]}')
    # Have the model judge:
    judge_score = faithfulness(client, q, res['answer'], res['contexts'])['score']
    print(f'\n  LLM-judge faithfulness: {judge_score:.2f}')
    print('  Your manual judgment (0.0 to 1.0): MANUAL_SCORE_HERE')
    manual_scores.append({'q': q, 'judge': judge_score, 'human': None})

# Update manually:
# manual_scores[0]['human'] = 1.0
# manual_scores[1]['human'] = 0.6
# manual_scores[2]['human'] = 0.8

todo3_reflection = """
[YOUR REFLECTION]

- Where did LLM judge agree with you? Disagree?
Q2 (ML frameworks): LJM judge 0.88, I'd score 1.00. Judge slightly penalized the refusal; humans prefer correct refusals.

- A failure mode of LLM judges (be specific):
Over-penalizing refusals. When the model correctly says "I don't know", LJM judges score it lower than a human would, because they're trained on answering rather than refusing.
- For your project, how will you build trust in your eval signal?
1. Manually score 50 answers as ground truth
2. Run LJM judge on all answers
3. When LJM judge disagrees with retrieval confidence, investigate
"""
print(todo3_reflection)


  Hybrid: dense=13 sparse=1 -> fused=13
Q: What programming languages does this person know?
A: Based on the retrieved context, this person knows the following programming languages and related technologies:

- **Python** (with Django framework)
- **JavaScript** (with Angular framework)
- **HTML/CSS**
- **SQL** (PostgreSQL and Oracle)

They also have experience with **REST APIs (GraphQL)** and **AWS (Redshift, S3)**.

[sample_resume.pdf], [portfolio_notes.txt]
Sources: ['sample_resume.pdf', 'portfolio_notes.txt', 'portfolio_notes.txt', 'sample_resume.pdf']

  LLM-judge faithfulness: 0.88
  Your manual judgment (0.0 to 1.0): MANUAL_SCORE_HERE
  Hybrid: dense=13 sparse=1 -> fused=13
Q: What machine learning frameworks have they used?
A: I don't know — that information isn't in my sources. The retrieved context covers Giulia Gonzalez's skills and experience (Python/Django, PostgreSQL, JavaScript/Angular, REST APIs, AWS, D3.js, etc.), but does not mention any machine learning frameworks.
S

## Save reflection


In [13]:
_t1 = todo1_reflection.strip() if 'todo1_reflection' in dir() else '[TODO 1]'
_t2 = todo2_reflection.strip() if 'todo2_reflection' in dir() else '[TODO 2]'
_t3 = todo3_reflection.strip() if 'todo3_reflection' in dir() else '[TODO 3]'

summary = f'''### Part 1 - Reranker A/B

Mean context_precision: no-rerank = {sum(r["cp_no"] for r in rows)/len(rows):.3f}, with = {sum(r["cp_yes"] for r in rows)/len(rows):.3f}
Mean faithfulness:      no-rerank = {sum(r["f_no"] for r in rows)/len(rows):.3f},  with = {sum(r["f_yes"] for r in rows)/len(rows):.3f}

{_t1}

---

### Part 2 - Hallucination + Guardrail

{_t2}

---

### Part 3 - LLM Judge vs Human

{_t3}
'''

rf = append_to_reflection('07', 'Reranking & RAG Evaluation', summary, output_dir=outputs_dir)
print(f'Reflection saved: {rf}')
print(); tracker.report()


Reflection saved: ..\outputs\homework_reflection.md

API COST REPORT
Total API calls:     31
Total input tokens:  16,873
Total output tokens: 2,303
Total cost:          $0.0852

Last 5 calls:
  1. [21:05:42] sonnet -- 538in/63out -- $0.0026
  2. [21:05:53] sonnet -- 589in/65out -- $0.0027
  3. [21:13:14] sonnet -- 598in/99out -- $0.0033
  4. [21:13:35] sonnet -- 538in/64out -- $0.0026
  5. [21:13:47] sonnet -- 589in/64out -- $0.0027


## Notebook 07 Complete!

**Key takeaways:**
- A reranker is the most cost-effective quality lever in RAG. Default: yes.
- Faithfulness > everything. A hallucinated, fluent answer is the worst outcome.
- LLM judges are noisy but cheap; use them as a regression signal, not absolute truth.

**Next:** **Notebook 08 — Project Integration (Resume RAG agent)**
